# Stacks


## Topic overview

LIFO structure for balanced matching and monotonic patterns.

## Pattern-recognition rules

- Balanced-brackets validation.
- Monotonic stack for next-greater / next-smaller.
- Expression parsing.

## Common data structures

- `list` used as a stack
- `collections.deque`

## Standard complexity expectations

- Monotonic-stack problems are typically O(n).

## Common mistakes

- Popping from an empty stack.
- Confusing when to pop vs. push.

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

---

# Concept overview: augmenting a stack with auxiliary state

Before the Min Stack problem itself, the general idea it teaches.

## The tension

A stack gives $O(1)$ `push`, `pop`, and `top` because every one of those
touches only **one end** of the structure. Any question about the *whole*
contents — the minimum, the maximum, the sum — normally costs $O(n)$, because
answering it means walking every element.

Min Stack asks for a whole-contents question in $O(1)$. So the answer cannot
be computed at query time. It must already exist when the query arrives.

## The resolution

Compute the answer at **push** time and store it **at that depth**.

This works because a stack is not just a set of values, it is a sequence of
*states*, and popping restores the previous state exactly. If each frame
records the answer for the stack as it stood when that frame was pushed, then
popping automatically restores the correct earlier answer. Nothing has to be
recomputed or undone.

## When this applies

Ask two questions:

1. Can the new answer be derived in $O(1)$ from the pushed value and the
   previous answer?
2. Does `pop` need to restore the previous answer exactly?

If both are yes, store the answer per frame. `min` qualifies because

$$\text{min}(S \cup \{v\}) = \min(v,\ \text{min}(S))$$

so each new answer is one comparison away from the previous one. `max` and
running `sum` qualify for the same reason.

**Where it fails:** the *median* or "second smallest" cannot be maintained
this way, because those are not recoverable from a single previous value plus
the new one. That is a heap problem, not a stack-augmentation problem.

## The cost

Auxiliary state is paid for in space: $O(n)$ extra, one entry per frame. The
trade is deliberate — spend memory at push time to make an otherwise $O(n)$
query free.

## Relation to monotonic stacks

Both appear under "stacks," and they are easy to confuse:

| | Auxiliary-state stack (this) | Monotonic stack |
|---|---|---|
| Elements | Nothing is ever discarded | Elements are popped to maintain order |
| Extra data | One derived value per frame | None; the ordering *is* the information |
| Answers | A query about current contents | Next-greater / next-smaller relationships |
| Height | Always equals the main stack | Shrinks and grows independently |

`min_stack` does happen to be non-increasing from bottom to top, but that is a
*consequence* of how it is built, not an invariant being enforced. Nothing is
discarded to preserve it.


---

# Min Stack

## Metadata

- Source: NeetCode
- Problem URL: https://neetcode.io/problems/minimum-stack
- Difficulty: Medium
- Topic: Stacks
- Solution path: `../Data Structures & Algorithms/min-stack/submission-0.py`


## 1. Setup

We need a stack supporting four operations:

- `push(val)`: add `val` to the top.
- `pop()`: remove the top value.
- `top()`: return the top value.
- `getMin()`: return the minimum value currently stored.

Every operation must run in $O(1)$ time.

A normal stack handles `push`, `pop`, and `top` in $O(1)$, but finding the
minimum by scanning the whole stack would take $O(n)$. So we need to store
additional information.


## 2. Governing principle

Use two synchronized stacks:

- `stack`: stores every value.
- `min_stack`: stores the minimum value corresponding to each stack state.

When a value is pushed, we also push the new minimum onto `min_stack`.
Therefore both stacks always have the same length, and

$$\texttt{min\_stack[-1]}$$

is always the minimum of all values currently in `stack`.


## 3. Algorithm design

**`push(val)`** — push `val` onto the main stack. For the minimum stack:
if this is the first value, then `val` is automatically the minimum;
otherwise compare `val` with the previous minimum,

$$\text{new minimum} = \min(\text{val},\ \text{current minimum})$$

and push that result onto `min_stack`.

**`pop()`** — remove the top element from both stacks, because both entries
represent the same stack state.

**`top()`** — return the last element of the main stack.

**`getMin()`** — return the last element of the minimum stack.


## 4. Pseudocode

```text
class MinStack:
    initialize:
        stack = empty list
        min_stack = empty list

    push(val):
        append val to stack
        if min_stack is empty:
            append val to min_stack
        else:
            new_minimum = minimum(val, top of min_stack)
            append new_minimum to min_stack

    pop():
        remove top of stack
        remove top of min_stack

    top():
        return top of stack

    getMin():
        return top of min_stack
```


## 5. Python solution

In [ ]:
class MinStack:
    def __init__(self) -> None:
        """Initialize the main stack and its synchronized minimum stack."""
        self.stack: list[int] = []
        self.min_stack: list[int] = []

    def push(self, val: int) -> None:
        """Push val and record the minimum for the resulting stack state."""
        self.stack.append(val)
        if not self.min_stack:
            # The first value is automatically the current minimum.
            self.min_stack.append(val)
        else:
            # Preserve the smaller of val and the previous minimum.
            current_minimum = self.min_stack[-1]
            self.min_stack.append(min(val, current_minimum))

    def pop(self) -> None:
        """Remove the top value and its corresponding minimum state."""
        self.stack.pop()
        self.min_stack.pop()

    def top(self) -> int:
        """Return the top value without removing it."""
        return self.stack[-1]

    def getMin(self) -> int:
        """Return the minimum value currently in the stack."""
        return self.min_stack[-1]

## 6. Trace the example

Start with `stack = []` and `min_stack = []`.

| Operation | `stack` | `min_stack` | Returns | Why |
|---|---|---|---|---|
| `push(1)` | `[1]` | `[1]` | | Stack was empty, so `1` is the minimum |
| `push(2)` | `[1, 2]` | `[1, 1]` | | $\min(2, 1) = 1$ |
| `push(0)` | `[1, 2, 0]` | `[1, 1, 0]` | | $\min(0, 1) = 0$ |
| `getMin()` | `[1, 2, 0]` | `[1, 1, 0]` | `0` | `min_stack[-1]` |
| `pop()` | `[1, 2]` | `[1, 1]` | | Both stacks pop together |
| `top()` | `[1, 2]` | `[1, 1]` | `2` | `stack[-1]` |
| `getMin()` | `[1, 2]` | `[1, 1]` | `1` | `min_stack[-1]`, recovered for free |

The last line is the whole point: popping the minimum `0` did not destroy the
knowledge that `1` was the minimum beforehand, because that fact was stored at
its own depth.


In [ ]:
min_stack = MinStack()
min_stack.push(1)
min_stack.push(2)
min_stack.push(0)

assert min_stack.getMin() == 0
min_stack.pop()
assert min_stack.top() == 2
assert min_stack.getMin() == 1

print("Example passed.")

## 7. Why duplicate minimums are necessary

Suppose the input is `push(2)`, `push(1)`, `push(1)`. The stacks become:

```text
stack     = [2, 1, 1]
min_stack = [2, 1, 1]
```

After one `pop()`:

```text
stack     = [2, 1]
min_stack = [2, 1]
```

The minimum is still `1`, which is correct. Storing a minimum for *every*
stack state avoids special handling for duplicate minimum values. An
optimization that only records a new minimum when `val < min_stack[-1]`
would pop the stored `1` here and wrongly report `2`.


In [ ]:
# Duplicate-minimum check: popping one copy of the minimum must keep the other.
duplicates = MinStack()
for value in (2, 1, 1):
    duplicates.push(value)

duplicates.pop()
assert duplicates.getMin() == 1, "duplicate minimum was lost"

print("Duplicate minimums handled correctly.")

## 8. Complexity analysis

Each operation performs only a constant number of list operations:

| Operation | Time |
|---|---|
| `push` | $O(1)$ |
| `pop` | $O(1)$ |
| `top` | $O(1)$ |
| `getMin` | $O(1)$ |

For $n$ stored elements, both stacks can contain $n$ values, so

$$\text{Space complexity} = O(n)$$


## 9. Alternative: one stack of pairs

Each stack element can instead store `(value, minimum_at_this_point)`.


In [ ]:
class MinStackPairs:
    def __init__(self) -> None:
        self.stack: list[tuple[int, int]] = []

    def push(self, val: int) -> None:
        if not self.stack:
            current_minimum = val
        else:
            previous_minimum = self.stack[-1][1]
            current_minimum = min(val, previous_minimum)
        self.stack.append((val, current_minimum))

    def pop(self) -> None:
        self.stack.pop()

    def top(self) -> int:
        return self.stack[-1][0]

    def getMin(self) -> int:
        return self.stack[-1][1]

Both methods have the same complexity. The two-stack version is usually
easier to understand initially, while the pair-stack version keeps all state
in one data structure.

**Recommended solution:** synchronized main stack and minimum stack.


## Randomized cross-check

Confirm both implementations against a brute-force `min()` over a plain list.


In [ ]:
import random


def cross_check(stack_class, operations: int = 20_000, seed: int = 0) -> None:
    """Compare stack_class against a plain list using min() as ground truth."""
    random.seed(seed)
    subject = stack_class()
    reference: list[int] = []

    for _ in range(operations):
        if reference and random.random() < 0.4:
            subject.pop()
            reference.pop()
        else:
            value = random.randint(-10, 10)
            subject.push(value)
            reference.append(value)

        if reference:
            assert subject.top() == reference[-1]
            assert subject.getMin() == min(reference)


cross_check(MinStack)
cross_check(MinStackPairs)
print("Both implementations passed 20,000 randomized operations.")

## Pattern recognition

The reusable idea is **carrying an auxiliary value alongside each stack
frame**. Whenever a query must be answered in $O(1)$ but depends on the whole
current contents, ask whether the answer can be computed at push time and
stored at that depth. The same trick answers "max in stack" or "sum of stack"
by swapping `min` for `max` or `+`.

Related: this is *not* a monotonic stack. A monotonic stack discards elements
to keep an ordering invariant; here nothing is discarded, and `min_stack` is
merely non-increasing as a consequence of how it is built.

## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| | | |


## Visualizing the two stacks

Seeing both stacks side by side at each step makes the lockstep property
concrete. Helpers live in `src/visualizations/stacks.py`.

`min_stack_states` replays a list of operations and records both stacks after
each one. Operations are `("push", value)` or `("pop", None)`.


In [ ]:
import sys
from pathlib import Path

# Make the repo root importable as `src.*` when running from notebooks/.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.visualizations import (  # noqa: E402
    format_min_stack_trace,
    min_stack_states,
    plot_min_stack_states,
)

operations = [("push", 1), ("push", 2), ("push", 0), ("pop", None)]
states = min_stack_states(operations)

print(format_min_stack_trace(states))

The text table needs no dependencies. The plot below needs `matplotlib`
(`pip install matplotlib`); each step draws the main stack on the left and the
minimum stack on the right, with the boxes that `top()` and `getMin()` read
highlighted.


In [ ]:
import matplotlib.pyplot as plt

ax = plot_min_stack_states(states, title="Min Stack: both stacks stay in lockstep")
plt.tight_layout()
plt.show()

Read the figure left to right:

- The two columns are **always the same height** — that is what makes popping
  them together correct.
- At `push(2)` the min column repeats `1` rather than storing `2`. The
  auxiliary stack tracks the minimum *of the state*, not the pushed value.
- At `push(0)` both tops are `0`, since the new value is the new minimum.
- After `pop()` the min column's top is `1` again. That value was never
  recomputed — it had been sitting at depth 1 since `push(2)`.

The duplicate-minimum case is worth plotting too, since it is the one that
breaks the "only store strictly smaller values" shortcut.


In [ ]:
duplicate_states = min_stack_states([("push", 2), ("push", 1), ("push", 1), ("pop", None)])

print(format_min_stack_trace(duplicate_states))

ax = plot_min_stack_states(duplicate_states, title="Duplicate minimums: both copies are stored")
plt.tight_layout()
plt.show()

Both `1`s are stored. Popping one leaves the other, so `getMin()` still
returns `1`. Had the second `1` been skipped as "not strictly smaller," the
pop would have removed the only stored `1` and `getMin()` would wrongly
report `2`.

## Final takeaways

- A stack query about the whole contents can be made $O(1)$ by computing the
  answer at push time and storing it per frame.
- Popping both stacks together is what makes the earlier answer reappear for
  free — no recomputation, no undo logic.
- Store a value for *every* frame, including ties. The saving from skipping
  duplicates is small and the correctness risk is real.
- This is not a monotonic stack: nothing is discarded.
